# 在 GPU 上编程
- 作者：[Marc Lelarge](https://www.di.ens.fr/~lelarge/) - [@marc_lelarge](https://x.com/marc_lelarge)

```
    ╔═══════════════════════════════════════════════════════════╗
    ║                  GPU PROGRAMMING BASICS                   ║
    ║                                                           ║
    ║   CPU: Single Chef        GPU: Kitchen Brigade            ║
    ║                                                           ║
    ║      🧑‍🍳                    🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳                 ║
    ║                            🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳                 ║
    ║   Slow but                 🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳 🧑‍🍳                 ║
    ║   Versatile               Fast when tasks                 ║
    ║                           are parallelizable!             ║
    ║                                                           ║
    ║   Grid → Blocks → Threads → Warps                         ║
    ║   ┌─────────────────────────────────────┐                 ║
    ║   │ Grid (All Blocks)                   │                 ║
    ║   │  ┌──────┐  ┌──────┐  ┌──────┐       │                 ║
    ║   │  │Block │  │Block │  │Block │ ...   │                 ║
    ║   │  │  0   │  │  1   │  │  2   │       │                 ║
    ║   │  │ T T T│  │ T T T│  │ T T T│       │                 ║
    ║   │  │ T T T│  │ T T T│  │ T T T│       │                 ║
    ║   │  └──────┘  └──────┘  └──────┘       │                 ║
    ║   └─────────────────────────────────────┘                 ║
    ║                                                           ║
    ║   Numba: Low-level Control 🔧                             ║
    ║   Triton: High-level Magic ✨                             ║
    ╚═══════════════════════════════════════════════════════════╝
```

这个 notebook 通过动手练习教你 GPU 编程的基础。我们先从 [Numba](https://numba.pydata.org/) 开始，它是 Python 的即时（JIT）编译器，提供底层 GPU 控制；然后转到 [Triton](https://openai.com/index/triton/)，这是 OpenAI 的高层、类 Python 的 GPU 编程语言。

**学习方法：** 这个 notebook 强调交互式编码，尽量减少前置理论。通过动手来学习。如果卡住了，可以向你的聊天助手要提示，但不要要完整答案。

**参考来源：**
- Nvidia [CUDA 编程指南](https://docs.nvidia.com/cuda/cuda-programming-guide/01-introduction/programming-model.html)
- Sasha Rush 的 [GPU-Puzzles](https://github.com/srush/GPU-Puzzles)
- Stanford CS336 [作业 2](https://github.com/stanford-cs336/assignment2-systems)

在设置 `Runtime / Change runtime type` 中打开 GPU 模式，并关闭 `AI Assistance / Show AI-powered inline completions`。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflowr/notebooks/blob/master/ModuleGPU/GPU_programming_basics.ipynb)


In [ ]:
import numba
import numpy as np
from numba import cuda

In [ ]:
import warnings
warnings.filterwarnings(
    action="ignore", category=numba.NumbaPerformanceWarning, module="numba"
)

## GPU 核心概念

在开始写代码之前，先理解 GPU 的执行模型：

- **流式多处理器（SM）**：GPU 的计算单元（类比 CPU 核心）
- **线程（Thread）**：最小的执行单元（处理一个元素）
- **线程块（Block）**：保证在同一个 SM 上运行的一组线程（可以共享内存并同步）
- **网格（Grid）**：线程块组织成 1D、2D 或 3D 网格
- **线程束（Warp）**：线程块内，线程按 32 个一组组成 warp。一个 warp 中的所有线程同时执行同一条指令（SIMT：单指令多线程）。如果 warp 中有些线程在控制流分支上走了某个分支，而另一些没有，那么没走该分支的线程会被屏蔽，而走该分支的线程继续执行。例如，如果某个条件只对 warp 里一半的线程成立，那么另一半会被屏蔽掉，让活跃线程执行这些指令。下面展示了这种情况。当一个 warp 里的不同线程走了不同的代码路径时，有时称为 warp 发散（warp divergence）。因此，当 warp 内线程走相同的控制流路径时，GPU 的利用率最高。
<div>
<img src="https://docs.nvidia.com/cuda/cuda-programming-guide/_images/active-warp-lanes.png" width="700"/>
</div>

**心智模型：** 网格 → 线程块 → 线程 → 线程束

**编程方式：** 你从**线程的视角**写代码（每个线程各自做什么），然后指定线程如何组织：
1. **线程**组成**线程块**（1D、2D 或 3D）
2. **线程块**组成**网格**（1D、2D 或 3D）
3. GPU 调度器把线程块分发到各个流式多处理器（SM）上
4. 每个线程块内，线程自动按 32 个一组组成**线程束**来执行

**关键洞察：** 你只需要写一遍标量的线程代码；并行性来自启动成千上万个线程，在同一段代码上处理不同的数据。

让我们开始写代码！


### 谜题 1：Map（映射）

**目标：** 用并行线程给数组的每个元素加 10。


In [ ]:
def map_spec(a):
    return a + 10

# 数组大小
SIZE = 4

# 创建输入和输出数组
a = np.arange(SIZE, dtype=np.float32)  # [0, 1, 2, 3]
out = np.zeros(SIZE, dtype=np.float32)

map_spec(a)

**任务：** 用 Numba 实现，让每个线程恰好给数组的一个元素加 10。

**提示：** 用 `cuda.threadIdx.x` 获取当前线程在其线程块内的索引。


In [ ]:
# 定义 CUDA 核
@cuda.jit
def map_kernel(out, a):
    # 获取线程索引
    i = cuda.threadIdx.x
    # 每个线程给一个元素加 10
    # 在这里写你的代码


# 把数组复制到 GPU
a_device = cuda.to_device(a)
out_device = cuda.to_device(out)

# kernel[grid, block](args)
# 启动核：grid = 1 个线程块，block = SIZE 个线程
map_kernel[1, SIZE](out_device, a_device)

# 把结果复制回 CPU
result = out_device.copy_to_host()

# 验证结果
expected = map_spec(a)
print(f"Input:    {a}")
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

### 谜题 2：向量加法

**目标：** 用并行线程逐元素地把两个向量相加。


In [ ]:
def zip_spec(a, b):
    return a + b

out = np.zeros(SIZE)
a = np.arange(SIZE)
b = np.arange(SIZE)
zip_spec(a,b)

In [ ]:
# 定义 CUDA 核
@cuda.jit
def zip_kernel(out, a, b):
    # 获取线程索引
    i = cuda.threadIdx.x
    # 在这里写你的代码

# 把向量移到设备上的函数
def init_pb(a=a, b=b, out=out):
    a_device = cuda.to_device(a)
    b_device = cuda.to_device(b)
    out_device = cuda.to_device(out)
    return a_device, b_device, out_device

a_device, b_device, out_device = init_pb()

# 启动核：1 个线程块，SIZE 个线程
zip_kernel[1, SIZE](out_device, a_device, b_device)

# 把结果复制回 CPU
result = out_device.copy_to_host()

# 验证结果
expected = zip_spec(a, b)
print(f"Input a:  {a}")
print(f"Input b:  {b}")
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

**实验：** 如果你启动的线程数超过数组大小，会发生什么？


In [ ]:
a_device, b_device, out_device = init_pb()

NUM_TRHEADS = 2*SIZE
zip_kernel[1, NUM_TRHEADS](out_device, a_device, b_device)

# 把结果复制回 CPU
result = out_device.copy_to_host()

# 验证结果
expected = zip_spec(a, b)
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

**结果：** 仍然能跑，但不安全！多余的线程会访问越界内存，可能导致崩溃或静默的数据损坏。

**任务：** 加一个边界保护（guard clause），防止线程访问超出数组边界的内存。


In [ ]:
# 带边界保护的 CUDA 核
@cuda.jit
def zip_guard_kernel(out, a, b, size):
    # 获取线程索引
    i = cuda.threadIdx.x
    # 在这里写你的代码

a_device, b_device, out_device = init_pb()

NUM_TRHEADS = 2*SIZE
zip_guard_kernel[1, NUM_TRHEADS](out_device, a_device, b_device, SIZE)

# 把结果复制回 CPU
result = out_device.copy_to_host()

# 验证结果
expected = zip_spec(a, b)
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

### 谜题 3：2D 矩阵

**目标：** 用 2D 线程块对 2D 矩阵做 map 操作。


In [ ]:
a = np.arange(SIZE * SIZE).reshape((SIZE, SIZE))
out = map_spec(a)
out

**关键洞察：** 线程块可以组织成 2D 或 3D 形状，这简化了把线程映射到 2D/3D 数据结构的过程。

**任务：** 用 2D 线程块，让每个线程处理矩阵的一个元素。

**提示：** 用 `cuda.threadIdx.x` 和 `cuda.threadIdx.y` 获取两个坐标。


In [ ]:
@cuda.jit
def map_2d_kernel(out, a, size):
    i = cuda.threadIdx.x
    j = cuda.threadIdx.y
    # 在这里写你的代码

a_device, b_device, out_device = init_pb(a=a, out=np.zeros_like(out))

TRHEAD_BLOCK = (SIZE, SIZE)
map_2d_kernel[1, TRHEAD_BLOCK](out_device, a_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = map_spec(a)
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

### 谜题 4：广播

**目标：** 用广播把两个向量相加（列 + 行 → 矩阵）。


In [ ]:
a = np.arange(SIZE).reshape(SIZE, 1)
b = np.arange(SIZE).reshape(1, SIZE)
out = a + b
out

In [ ]:
@cuda.jit
def broadcast_kernel(out, a, b, size):
    i = cuda.threadIdx.x
    j = cuda.threadIdx.y
    # 在这里写你的代码
    

a_device, b_device, out_device = init_pb(a=a, b=b, out=np.zeros_like(out))

THREAD_BLOCK = (2*SIZE, 3*SIZE)
broadcast_kernel[1, THREAD_BLOCK](out_device, a_device, b_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = a + b
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

**新概念：** 到目前为止我们只用过**线程块**。现在让我们用**网格**这个维度。

和线程块一样，网格也可以是 1D、2D 或 3D。

**任务：** 用 2D 网格、每个线程块只有 1 个线程，来计算广播加法。

**提示：** 用 `cuda.blockIdx.x` 和 `cuda.blockIdx.y` 获取线程块在网格中的位置。


In [ ]:
@cuda.jit
def broadcast_grid_kernel(out, a, b, size):
    i = cuda.blockIdx.x 
    j = cuda.blockIdx.y
    # 在这里写你的代码


a_device, b_device, out_device = init_pb(a=a, b=b, out=np.zeros_like(out))

# 每块 1 个线程，2D 网格
THREADS = 1
GRID = (SIZE, SIZE)
broadcast_grid_kernel[GRID, THREADS](out_device, a_device, b_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = a + b
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

**下一个挑战：** 用 1D 网格和 1D 线程块实现同样的操作。

**提示：** 你需要从 1D 的块索引和线程索引计算出 2D 索引 (i, j)。


In [ ]:
@cuda.jit
def broadcast_grid_kernel(out, a, b, size):
    # 在这里写你的代码
    

a_device, b_device, out_device = init_pb(a=a, b=b, out=np.zeros_like(out))

THREADS = SIZE
GRID = SIZE
broadcast_grid_kernel[GRID, THREADS](out_device, a_device, b_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = a + b
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

**最终挑战：** 用 2D 网格配 2D 线程块。

这里我们配置 `cuda.blockDim.x = cuda.blockDim.y = 2`（每个块有 2×2=4 个线程）和 `SIZE//2 = 2`（我们有 2×2=4 个块）。

**理解索引：** 输出中的每个元素 (i, j) 由块索引和线程索引组合得到：

| blockIdx.x | blockIdx.y | threadIdx.x | threadIdx.y | **i** | **j** | Computes |
|------------|------------|-------------|-------------|-------|-------|----------|
| 0 | 0 | 0 | 0 | **0** | **0** | out[0,0] |
| 0 | 0 | 1 | 0 | **1** | **0** | out[1,0] |
| 0 | 0 | 0 | 1 | **0** | **1** | out[0,1] |
| 0 | 0 | 1 | 1 | **1** | **1** | out[1,1] |
| 1 | 0 | 0 | 0 | **2** | **0** | out[2,0] |
| 1 | 0 | 1 | 0 | **3** | **0** | out[3,0] |
| 0 | 1 | 0 | 0 | **0** | **2** | out[0,2] |
| 0 | 1 | 1 | 1 | **1** | **3** | out[1,3] |
| 1 | 1 | 0 | 0 | **2** | **2** | out[2,2] |
| 1 | 1 | 1 | 1 | **3** | **3** | out[3,3] |

**公式：** `i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x`（j 类似）


In [ ]:
@cuda.jit
def broadcast_grid_kernel(out, a, b, size):
    # 在这里写你的代码
    

a_device, b_device, out_device = init_pb(a=a, b=b, out=np.zeros_like(out))

THREADS = (SIZE//2 , SIZE//2)
GRID = (SIZE//2, SIZE//2)  
broadcast_grid_kernel[GRID, THREADS](out_device, a_device, b_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = a + b
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

## GPU 内存层次结构

<div>
<img src="https://docs.nvidia.com/cuda/cuda-programming-guide/_images/gpu-cpu-system-diagram.png" width="700"/>
</div>

**主要的内存类型：**

- **全局内存（DRAM）**：容量大但慢（约几百个周期的延迟）
  - GPU 上所有 SM 都能访问
  - 系统内存（主机 DRAM）更慢（需要 PCIe 传输）

- **共享内存**：容量小但快（约 1 个周期的延迟）
  - 芯片上的内存，由线程块内的线程共享
  - 由程序员管理的缓存（由你控制加载什么）
  - 容量有限（通常每个 SM 48-164 KB）

- **寄存器**：最快，每个线程私有
  - 线程直接访问（不需要 load/store）
  - 非常有限（寄存器溢出会导致性能下降）

**性能策略：** 把共享内存作为手动管理的缓存，尽量减少全局内存访问。


### 谜题 5：池化（滑动窗口求和）

**目标：** 计算滑动窗口和：每个输出元素是至多 3 个输入元素的和（当前元素和前 2 个元素）。


In [ ]:
def pool_spec(a):
    out = np.zeros(a.shape)
    for i in range(a.shape[0]):
        out[i] = a[max(i - 2, 0) : i + 1].sum()
    return out

SIZE = 8
a = np.arange(SIZE)
out = pool_spec(a)
out

In [ ]:
@cuda.jit
def pool_kernel(out, a, size):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < size:
        # 手动计算求和 - CUDA 里不能用切片！
        temp_sum = 0.0
        for k in range(max(i - 2, 0), i + 1):
            temp_sum += a[k]  # 使用全局内存
        out[i] = temp_sum

a_device, b_device, out_device = init_pb(a=a, out=np.zeros_like(out))

THREADS = SIZE//2
GRID = (2,1)  
pool_kernel[GRID, THREADS](out_device, a_device, SIZE)

result = out_device.copy_to_host()

# 验证结果
expected = pool_spec(a)
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")
r = 3
print(f"number of access to global memory: 1 + 2 + {SIZE-2} x {r} reads -> {1+2+(SIZE-2)*r} global reads in total")

**问题：** 朴素实现会多次访问全局内存（SIZE=8 时需要 21 次读）。

**解决方案：** 用共享内存把全局内存访问减少到总共 12 次读（SIZE=8 时）。

**内存层次结构可视化：**
```
┌─────────────────────────────────────┐
│  全局内存 (a, out)                  │  ← 慢，所有线程都能访问
│  - 高延迟（约几百个周期）            │
│  - 大容量（GB 级）                  │
└─────────────────────────────────────┘
         ↓                    ↓
   ┌─────────┐          ┌─────────┐
   │ Block 0 │          │ Block 1 │
   │ 共享内存 │          │ 共享内存 │      ← 快，只在线程块内可访问
   │ (快)    │          │ (快)    │         （约 1 个周期）
   └─────────┘          └─────────┘
```

**原理 - Halo 加载：**
```
全局:      [0, 1, 2, 3, 4, 5, 6, 7]
Block 0 加载:              Block 1 加载:
        ↓                          ↓
共享: [0, 0, 0, 1, 2, 3] 共享: [2, 3, 4, 5, 6, 7]
         └─┘  └──────────┘         └──┘  └──────────┘
        halo   主数据               halo   主数据
        (边界填充)                  (与 Block 0 重叠)
```

**关键约束：**
1. 共享内存的大小必须是**编译期常量**（不能是运行时变量）
2. 加载共享内存后，调用 `cuda.syncthreads()` 确保所有线程在使用数据之前都能看到它

**任务：** 用带 halo 区的共享内存实现池化核。


In [ ]:
TPB = 4  # TPB = 4  # 每个线程块的线程数
SharedMem = TPB + 2  # 不能在运行时计算
@cuda.jit
def pool_kernel_shared(out, a, size):
    # 分配带 HALO 的共享内存（为边界多出的元素）
    # 需要 TPB + 2 个额外元素（用于 2 元素回看）
    shared = cuda.shared.array(SharedMem, numba.float32)
    # 在这里写你的代码

a_device, b_device, out_device = init_pb(a=a, out=np.zeros_like(out))


GRID = (SIZE // TPB, 1)  # SIZE=8, TPB=4 时为 (2, 1)
pool_kernel_shared[GRID, TPB](out_device, a_device, SIZE)

result = out_device.copy_to_host()

expected = pool_spec(a)
print(f"Output:   {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

### 谜题 6：点积（并行归约）

**目标：** 计算两个向量的点积：`dot(a, b) = sum(a[i] * b[i])`

**挑战：** 串行代码很简单，但并行归约很复杂，因为我们需要**合并多个线程的结果**。


In [ ]:
def dot_spec(a,b):
    tot = 0
    for i in range(len(a)):
        tot += a[i]*b[i]
    return tot

SIZE = 8
a = np.arange(SIZE, dtype=np.float32)
b = np.arange(SIZE, dtype=np.float32)
dot_spec(a,b)

**解决方案思路：** 在每个线程块内做基于树的归约，然后跨线程块用原子加法。

**可视化 - 树状归约：**

![Tree-based parallel sum](https://www.cs.uaf.edu/2012/fall/cs441/lecture/tree_sum_16td.png)

**原理：**
1. 每个线程计算一个逐元素乘积：`a[i] * b[i]`
2. 把结果存到共享内存
3. 用 log2(n) 步归约：步长 = n/2, n/4, n/8, ..., 1
4. 线程 0 把线程块的局部和写到输出


In [ ]:
def dot_tree(a,b):
    size = len(a)
    shared_mem = np.zeros(size)
    for i in range(size):
       shared_mem[i] = a[i]*b[i]
    stride = size // 2
    while stride > 0:
        for i in range(stride):
            shared_mem[i] += shared_mem[i+stride]
        stride //=2
    return shared_mem[0]
dot_tree(a,b)       

**任务：** 用 Numba 实现基于树的点积，配置如下：
- **每个线程块 256 个线程**（固定）
- **多个线程块**覆盖输入大小（自动计算）

**提示：** 线程块内归约之后，用 `cuda.atomic.add()` 安全地累加所有线程块的局部和。


In [ ]:
SIZE = 800
a = np.arange(SIZE, dtype=np.float32)
b = np.arange(SIZE, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (SIZE + threads_per_block - 1) // threads_per_block
print(f"threads per block: {threads_per_block}")
print(f"blocks per grid: {blocks_per_grid}")

对每个线程块，你可以这样实现基于树的求和：先创建一个大小为 256 的共享内存，存放与该线程块线程关联的 `a[i]*b[i]`，然后在块内求和。最后一步是累加所有中间结果：每个线程块加上自己的结果。这最后一步，你可能想用 `cuda.atomic.add`，见下面：


### 理解原子操作

**为什么需要原子操作？**

当多个线程写入同一个内存位置时，非原子操作会因**竞态条件**而丢失更新。

**没有原子操作的问题：**

当你写 `out[0] = out[0] + value` 时，它其实是 3 个独立步骤：
```python
# out[0] = out[0] + value 分解为：
1. 读:   temp = out[0]      # 读取当前值
2. 改: temp = temp + value # 加上去
3. 写:  out[0] = temp       # 写回
```

**竞态条件示例：**
```
初始: out[0] = 0

线程 A (Block 0):              线程 B (Block 1):
1. 读: temp_A = 0
2. 改: temp_A = 0 + 5
                                1. 读: temp_B = 0      ← 仍然看到 0！
3. 写: out[0] = 5
                                2. 改: temp_B = 0 + 3 ← 用的是旧值！
                                3. 写: out[0] = 3      ← 覆盖了 5！

最终: out[0] = 3  ❌ 应该是 8！
```

**原子操作做什么：**

`cuda.atomic.add(out, 0, value)` 在**整个读-改-写**期间**锁定该内存位置**：

```
初始: out[0] = 0

线程 A (Block 0):              线程 B (Block 1):
🔒 锁定 out[0]
1. 读: temp_A = 0
2. 改: temp_A = 0 + 5
3. 写: out[0] = 5
🔓 解锁 out[0]
                                🔒 锁定 out[0]  ← 必须等解锁
                                1. 读: temp_B = 5      ← 看到更新后的值！
                                2. 改: temp_B = 5 + 3
                                3. 写: out[0] = 8
                                🔓 解锁 out[0]

最终: out[0] = 8  ✅ 正确！
```

**要点：** 当多个线程（来自同一个或不同线程块）更新同一个内存位置时，使用原子操作。


In [ ]:
@cuda.jit
def dot_kernel_numba(a, b, out, size):
    shared = cuda.shared.array(256, numba.float32)
    # 在这里写你的代码
    

expected = np.dot(a, b)
a_device, b_device, out_device = init_pb(a=a, b=b, out=np.zeros_like([expected]))

size = a_device.shape[0]
dot_kernel_numba[blocks_per_grid, threads_per_block](a_device, b_device, out_device, size)

result = out_device.copy_to_host()
print(f"CUDA result: {result[0]}")
print(f"NumPy result: {expected}")
print(f"Match: {np.allclose(result[0], expected)}")

## Numba vs Triton：概念对比

### Numba CUDA：网格和线程块维度

**关键概念：**
- `kernel[grid, block](args)` - 启动语法
- **网格** = `(blocks_x, blocks_y, blocks_z)` - 有多少个线程块
- **线程块** = `(threads_x, threads_y, threads_z)` - 每个线程块的线程数
- **总线程数** = `grid_x × grid_y × grid_z × block_x × block_y × block_z`
- **手动索引**：你用 `blockIdx`、`threadIdx`、`blockDim` 计算索引


### Triton：程序网格（更高层的抽象）

在 **Triton** 中，你指定一个**程序网格**并使用**程序 ID**。Triton 自动处理底层线程。

**关键概念：**
- `kernel[grid](args, BLOCK_SIZE=...)` - 启动语法
- **网格** = `(programs_x, programs_y, programs_z)` - 程序实例的数量
- **没有显式的线程维度** - Triton 自动向量化
- **用数据块工作**：使用 `tl.arange()` 和向量化操作

---

### 对比表

| 方面 | Numba CUDA | Triton |
|--------|------------|--------|
| **启动语法** | `kernel[grid, block](args)` | `kernel[grid](args, BLOCK=...)` |
| **网格表示** | **线程块**的数量 | **程序**的数量 |
| **线程块/线程控制** | 显式：每个块 `(tx, ty, tz)` | 抽象化：在数据块上工作 |
| **线程索引** | 手动：`blockIdx`、`threadIdx` | 自动：`tl.program_id()` + `tl.arange()` |
| **典型网格** | `(n_blocks_x, n_blocks_y, n_blocks_z)` | `(n_programs_x, n_programs_y, n_programs_z)` |
| **典型线程块** | `(threads_x, threads_y, threads_z)` | 无（隐含在 `BLOCK_SIZE` 中） |
| **内存访问** | 每线程标量索引 | 向量化的块操作 |
| **抽象级别** | 底层（像 CUDA C） | 高层（编译器优化） |
| **同步** | 显式：`cuda.syncthreads()` | 大多自动 |

---

### 要点

- **Numba CUDA**：你从**线程块**的角度思考（两层层次：网格 → 线程块 → 线程）
- **Triton**：你从**作用于数据块的程序**的角度思考（单层：网格 → 程序，自动向量化）

**心智模型：** 每个 Triton 程序 ≈ 一个 CUDA 线程块，但 Triton 自动把线程级工作向量化。

---

### 实践例子：向量加法

下面是前面某个 Numba 谜题的解答：


In [ ]:
# 带边界保护的 CUDA 核
@cuda.jit
def zip_guard_kernel(out, a, b, size):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < size:
        out[i] = a[i] + b[i]

SIZE = 1000
out = np.zeros(SIZE)
a = np.arange(SIZE)
b = np.arange(SIZE)

a_device, b_device, out_device = init_pb(a=a, b=b, out=out)

threads_per_block = 256
blocks_per_grid = (SIZE + threads_per_block - 1) // threads_per_block
zip_guard_kernel[blocks_per_grid, threads_per_block](out_device, a_device, b_device, SIZE)

# 把结果复制回 CPU
result = out_device.copy_to_host()

# 验证结果
expected = zip_spec(a, b)
print(f"Correct:  {np.allclose(result, expected)}")

### Triton 实现：向量加法

**与 Numba 的关键区别：**
- 直接操作 PyTorch 张量（不需要手动管理内存）
- 每个程序处理多个元素（向量化）
- `BLOCK_SIZE` 是编译期常量，便于优化


In [ ]:
import triton
import triton.language as tl
import torch
from einops import rearrange

def get_device(index: int = 0) -> torch.device:
    """Try to use the GPU if possible, otherwise, use CPU."""
    if torch.cuda.is_available():
        return torch.device(f"cuda:{index}")
    else:
        return torch.device("cpu")

In [ ]:
@triton.jit
def zip_guard_triton(a_ptr, b_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    # Triton 用 program_id（程序 id，相当于块索引）而不是显式的 blockIdx/threadIdx
    # 每个"程序"一次处理 BLOCK_SIZE 个元素（向量化）
    pid = tl.program_id(0)
    
    # Triton 为一个*向量*（BLOCK_SIZE 个元素）计算偏移
    # 和 Numba 中每个线程处理 1 个元素不同，
    # Triton 每个程序实例处理多个元素
    offset = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    
    # Triton 用基于掩码的边界保护来做向量化操作
    # 而不是标量的 if 语句（if i < size）
    mask = offset < n
    
    # 向量化加载：带掩码一次加载 BLOCK_SIZE 个元素
    # Numba 是标量加载：a[i]
    a = tl.load(a_ptr + offset, mask=mask)
    b = tl.load(b_ptr + offset, mask=mask)
    
    # 向量化计算（和 Numba 一样，但作用在向量上）
    c = a + b
    
    # 带掩码的向量化存储（对比 Numba 的标量存储）
    tl.store(out_ptr + offset, c, mask=mask)


# Triton 直接操作 PyTorch 张量（不需要手动 copy_to_host）
# Numba 需要显式的设备内存管理（init_pb、copy_to_host）
a = torch.randn(SIZE, device=get_device())
b = torch.randn(SIZE, device=get_device())
out = torch.empty_like(a)


# 启动语法区别：
# - BLOCK_SIZE 是编译期常量（tl.constexpr），便于优化
# - 只需要指定网格维度（不需要 threads_per_block）
# - Triton 在每个程序内部自动向量化
BLOCK_SIZE = 256
grid = (triton.cdiv(SIZE, BLOCK_SIZE),)  # 只有网格大小，没有块大小
zip_guard_triton[grid](a, b, out, SIZE, BLOCK_SIZE=BLOCK_SIZE)

expected = zip_spec(a, b)
print(f"Correct:  {np.allclose(out.cpu().numpy(), expected.cpu().numpy())}")

### Triton 点积

**任务：** 在 Triton 中实现点积。

**有用的函数：**
- [`tl.sum()`](https://triton-lang.org/main/python-api/generated/triton.language.sum.html) - 线程块内的并行归约
- [`tl.atomic_add()`](https://triton-lang.org/main/python-api/generated/triton.language.atomic_add.html) - 跨程序的原子加法


In [ ]:
@triton.jit
def dot_kernel(
    a_ptr,  # 指向第一个输入向量的指针
    b_ptr,  # 指向第二个输入向量的指针
    out_ptr,  # 指向输出标量的指针
    size,  # 向量大小
    BLOCK_SIZE: tl.constexpr,  # 每个程序处理的元素数
):
    # 程序 ID（类似于 blockIdx.x）
    pid = tl.program_id(0)
    
    # 计算这个程序的数据块的偏移
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    
    # 处理边界的掩码
    mask = offsets < size
    
    # 加载数据
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    
    # 逐元素乘法
    products = a * b
    
    # 在这个块内做归约求和（自动并行归约！）
    block_sum = tl.sum(products)
    
    # 原子加进输出（每个线程块只有一个线程做这个）
    tl.atomic_add(out_ptr, block_sum)


def dot_triton(a, b):
    """Wrapper function to launch the kernel"""
    # 分配输出
    out = torch.zeros(1, device=a.device, dtype=a.dtype)
    
    # 网格和块配置
    size = a.shape[0]
    BLOCK_SIZE = 256  
    grid = (triton.cdiv(size, BLOCK_SIZE),)
    
    # 启动核
    dot_kernel[grid](a, b, out, size, BLOCK_SIZE=BLOCK_SIZE)
    
    return out


# 使用示例
SIZE = 10
a = torch.arange(SIZE, dtype=torch.float32, device=get_device())
b = torch.arange(SIZE, dtype=torch.float32, device=get_device())

result_triton = dot_triton(a, b)
result_torch = torch.dot(a, b)

print(f"Triton result: {result_triton.item()}")
print(f"PyTorch result: {result_torch.item()}")
print(f"Match: {torch.allclose(result_triton, result_torch)}")

### Triton Softmax

**目标：** 对一批形状为 `(batch_size, dim)` 的向量 `z` 实现 softmax：`torch.softmax(z, dim=1)`

**关键概念 - 张量步长（Strides）：**

步长定义了如何在连续内存中导航多维张量。

**可视化：**
```
内存: [a, b, c, d, e, f, g, h, i, j, k, l]
形状 (3, 4)，步长 (4, 1):
  [[a, b, c, d],    ← 到下一行跳 4 个，到下一列跳 1 个
   [e, f, g, h],
   [i, j, k, l]]
```

**为什么重要：** 要在 softmax 中处理一行，我们需要：
1. 找到起始地址：`row_start_ptr = base_ptr + row_idx * row_stride`
2. 用列步长加载该行的所有元素


In [ ]:
# 连续的 2D 张量（3×4）
x = torch.randn(3, 4)
x.stride()  # (4, 1)
# - 移到下一行：跳 4 个元素
# - 移到下一列：跳 1 个元素

In [ ]:
# 转置（现在是 4×3）
y = x.t()
y.stride()  # (1, 4)
# - 移到下一行：跳 1 个元素（原来是列）
# - 移到下一列：跳 4 个元素（原来是行）
# 注意：y 和 x 共享内存，只是访问模式不同

**实现策略：**
- 每个程序独立处理一整行
- 每个程序对自己那行的所有列做 softmax
- 用 row_stride 定位每一行的起始地址


In [ ]:
@triton.jit
def triton_softmax_kernel(x_ptr, y_ptr, x_row_stride, y_row_stride, num_cols, BLOCK_SIZE: tl.constexpr):
    assert num_cols <= BLOCK_SIZE
    # 每个行独立处理
    # 在这里写你的代码

def triton_softmax(x: torch.Tensor):
    x = x.contiguous()
    # 分配输出张量
    y = torch.empty_like(x)
    # 确定网格
    M, N = x.shape  # 行数 x 列数
    block_size = triton.next_power_of_2(N)  # 每个块包含所有列
    num_blocks = M  # 每个块是一行
    # 启动核
    triton_softmax_kernel[(M,)](
        x_ptr=x, y_ptr=y,
        x_row_stride=x.stride(0), y_row_stride=y.stride(0),
        num_cols=N, BLOCK_SIZE=block_size
    )
    return y

In [ ]:
torch.manual_seed(0)
x = torch.randn(1823, 781, device=get_device())
y_triton = triton_softmax(x)
y_torch = torch.softmax(x, axis=1)
assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)

In [ ]:
y_triton = triton_softmax(x.t())
y_torch = torch.softmax(x.t(), axis=1)
assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)

In [ ]:
DEVICE = get_device()

@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],  # 作为绘图 x 轴的参数名
        x_vals=[128 * i for i in range(2, 100)],  # `x_name` 的不同取值
        line_arg='provider',  # 取值对应图中不同线条的参数名
        line_vals=['triton', 'torch'],  # `line_arg` 的取值
        line_names=["Triton", "Torch"],  # 线条的标签
        styles=[('blue', '-'), ('green', '-')],  # 线条样式
        ylabel="GB/s",  # y 轴标签
        plot_name="softmax-performance",  # 图名，也用作保存图片的文件名
        args={'M': 4096},  # 不在 `x_names` 和 `y_name` 中的函数参数值
    ))

def benchmark(M, N, provider):
    x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
    stream = getattr(torch, DEVICE.type).Stream()
    getattr(torch, DEVICE.type).set_stream(stream)
    if provider == 'torch':
        ms = triton.testing.do_bench(lambda: torch.softmax(x, axis=-1))
    if provider == 'triton':
        ms = triton.testing.do_bench(lambda: triton_softmax(x))
    gbps = lambda ms: 2 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms)


benchmark.run(show_plots=True, print_data=False)

## Triton 块指针（进阶）

**块指针**是 Triton 面向分块内存访问的高层抽象，消除了容易出错的手动指针运算。

**核心概念：** 与其手动计算 `ptr + offset`，块指针封装了：
- **你在张量中的位置**（偏移量）
- **你要访问的块**（block_shape）
- **如何导航内存**（步长）

---

### 例子：2D 张量块指针

```python
x_block_ptr = tl.make_block_ptr(
    x_ptr,                                    # 基地址
    shape=(ROWS, D),                          # 完整张量：ROWS × D
    strides=(x_stride_row, x_stride_dim),     # 行/列之间的跳跃
    offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),  # 从第 row_tile 行、第 0 列开始
    block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),   # 块大小
    order=(1, 0),                             # 行主序布局
)
```

**可视化：**
```
完整张量 (ROWS × D):
┌─────────────────────────────────┐
│ [0,0]  ......  [0, D-1]         │ ← row_tile_idx=0 加载 ROWS_TILE_SIZE 行
├─────────────────────────────────┤
│ [ROWS_TILE_SIZE, 0] ...         │ ← row_tile_idx=1
├─────────────────────────────────┤
│  ...                            │
└─────────────────────────────────┘
    └─ D_TILE_SIZE ─┘  (块宽度)
```

---

### 例子：1D 张量块指针

```python
weight_block_ptr = tl.make_block_ptr(
    weight_ptr,
    shape=(D,),                    # 1D 向量
    strides=(weight_stride_dim,),  # 元素间距
    offsets=(0,),                  # 从开头开始
    block_shape=(D_TILE_SIZE,),    # 加载 D_TILE_SIZE 个元素
    order=(0,),                    # 1D 排序
)
```

---

### 使用模式：分块计算

```python
# 初始化累加器
output = tl.zeros((ROWS_TILE_SIZE,), dtype=tl.float32)

# 按块循环 D 维度
for i in range(tl.cdiv(D, D_TILE_SIZE)):
    # 加载当前块，自动做边界检查
    row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero")
    # 形状: (ROWS_TILE_SIZE, D_TILE_SIZE)

    weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero")
    # 形状: (D_TILE_SIZE,)

    # 计算加权和：对每一行按列求和
    output += tl.sum(row * weight[None, :], axis=1)  # 累加

    # 前进到下一个块
    x_block_ptr = tl.advance(x_block_ptr, (0, D_TILE_SIZE))      # 向右移
    weight_block_ptr = tl.advance(weight_block_ptr, (D_TILE_SIZE,))

# 写结果
tl.store(output_block_ptr, output, boundary_check=(0,))
```

---

### 关键特性

1. **自动边界检查：** `boundary_check=(0, 1)` 处理不能完美整除的块
   - 第 0 维（行）：可能不能被 `ROWS_TILE_SIZE` 整除
   - 第 1 维（列）：可能不能被 `D_TILE_SIZE` 整除


### 把 Triton 核与 PyTorch 集成

**例子：** 用块指针做高效的加权求和分块计算。


In [ ]:
def weighted_sum(x, weight):
    # 这里假设 x 的形状是 n 维 [..., D]，weight 是 1D 形状 [D]
    return (weight * x).sum(axis=-1)



@triton.jit
def weighted_sum_fwd(
    x_ptr, weight_ptr,  # 输入指针
    output_ptr,  # 输出指针
    x_stride_row, x_stride_dim,  # 步长告诉我们如何在张量的每个轴上移动一个元素
    weight_stride_dim,  # 通常为 1
    output_stride_row,  # 通常为 1
    ROWS, D,
    ROWS_TILE_SIZE: tl.constexpr, D_TILE_SIZE: tl.constexpr,  # 块形状必须在编译期已知
):
    # 每个实例计算 x 的一个行块的加权和。
    # `tl.program_id` 让我们知道当前运行在哪个线程块
    row_tile_idx = tl.program_id(0)
    
    # 块指针让我们可以从一个 ND 内存区域中选择
    # 并移动我们的选区。
    # 块指针必须知道：
    # - 指向张量第一个元素的指针
    # - 张量的整体形状，用于处理越界访问
    # - 每个维度的步长，以便正确使用内存布局
    # - 起始块的 ND 坐标，也就是"偏移量"
    # - 每次 load/store 所用的块形状
    # - 内存中各维度从主到次的顺序
    # 轴（= np.argsort(strides)），用于优化，在 H100 上尤其有用
    
    x_block_ptr = tl.make_block_ptr(
        x_ptr,
        shape=(ROWS, D),
        strides=(x_stride_row, x_stride_dim),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )
    
    weight_block_ptr = tl.make_block_ptr(
        weight_ptr,
        shape=(D,),
        strides=(weight_stride_dim,),
        offsets=(0,),
        block_shape=(D_TILE_SIZE,),
        order=(0,),
    )
    
    output_block_ptr = tl.make_block_ptr(
        output_ptr,
        shape=(ROWS,),
        strides=(output_stride_row,),
        offsets=(row_tile_idx * ROWS_TILE_SIZE,),
        block_shape=(ROWS_TILE_SIZE,),
        order=(0,),
    )
    
    # 初始化一个要写入的缓冲区
    output = tl.zeros((ROWS_TILE_SIZE,), dtype=tl.float32)
    
    for i in range(tl.cdiv(D, D_TILE_SIZE)):
        # 加载当前的块指针
        # 由于 ROWS_TILE_SIZE 可能不整除 ROWS，D_TILE_SIZE 可能不整除 D，
        # 两个维度都需要边界检查
        row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero")  # (ROWS_TILE_SIZE, D_TILE_SIZE)
        weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero")  # (D_TILE_SIZE,)
        
        # 计算这一行的加权和。
        output += tl.sum(row * weight[None, :], axis=1)
        
        # 把指针移到下一个块。
        # 这些是 (行, 列) 的坐标增量
        x_block_ptr = x_block_ptr.advance((0, D_TILE_SIZE))  # 在最后一个维度上移动 D_TILE_SIZE
        weight_block_ptr = weight_block_ptr.advance((D_TILE_SIZE,))  # 移动 D_TILE_SIZE
    
    # 把输出写到输出块指针（每行一个标量）。
    # 由于 ROWS_TILE_SIZE 可能不整除 ROWS，需要边界检查
    tl.store(output_block_ptr, output, boundary_check=(0,))

In [ ]:
def weighted_sum_triton(x: torch.Tensor, weight: torch.Tensor):
    D = x.shape[-1]
    output_dims = x.shape[:-1]   
    # 把输入张量重塑成 2D
    
    x = rearrange(x, "... d -> (...) d")
    # 需要初始化空的输出张量。注意这些元素不一定为 0！
    y = torch.empty(x.shape[0], device=x.device)

    D_TILE_SIZE = triton.next_power_of_2(D) // 16  # 大约循环 16 次覆盖 embedding 维度
    ROWS_TILE_SIZE = 16  # 每个线程一次处理 16 个 batch 元素
        
    # 用 n 个实例在我们的 1D 网格中启动核。
    n_rows = y.numel()
    weighted_sum_fwd[(triton.cdiv(n_rows, ROWS_TILE_SIZE),)](
            x, weight,
            y,
            x.stride(0), x.stride(1),
            weight.stride(0),
            y.stride(0),
            ROWS=n_rows, D=D,
            ROWS_TILE_SIZE=ROWS_TILE_SIZE, D_TILE_SIZE=D_TILE_SIZE,
        )
        
    return y.view(output_dims)

In [ ]:
def check_equal3(f1, f2):
    x = torch.randn(64, 64, 2048, device=get_device())
    w = torch.randn(2048, device=get_device())
    y1 = f1(x,w)
    y2 = f2(x,w)
    assert torch.allclose(y1, y2, atol=1e-4)

In [ ]:
check_equal3(weighted_sum,weighted_sum_triton)

In [ ]:
class WeightedSumFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight):
        # 缓存 x 和 weight，供反向传播使用，那时
        # 我们只收到关于输出张量的梯度，
        # 需要计算关于 x 和 weight 的梯度。
        D, output_dims = x.shape[-1], x.shape[:-1]

        # 把输入张量重塑成 2D
        x_reshaped = rearrange(x, "... d -> (...) d")
        ctx.output_dims = output_dims

        ctx.save_for_backward(x, weight)

        assert len(weight.shape) == 1 and weight.shape[0] == D, "Dimension mismatch"
        assert x.is_cuda and weight.is_cuda, "Expected CUDA tensors"
        assert (
            x_reshaped.is_contiguous()
        ), "Our pointer arithmetic will assume contiguous x"

        D_TILE_SIZE = (
            triton.next_power_of_2(D) // 16
        )  # )  # 大约循环 16 次覆盖 embedding 维度
        ROWS_TILE_SIZE = 16  # 每个线程一次处理 16 个 batch 元素

        # 需要初始化空的输出张量。注意这些元素不一定为 0！
        y = torch.empty(x_reshaped.shape[0], device=x.device)

        # 用 n 个实例在我们的 1D 网格中启动核。
        n_rows = y.numel()
        weighted_sum_fwd[(triton.cdiv(n_rows, ROWS_TILE_SIZE),)](
            x_reshaped,
            weight,
            y,
            x_reshaped.stride(0),
            x_reshaped.stride(1),
            weight.stride(0),
            y.stride(0),
            ROWS=n_rows,
            D=D,
            ROWS_TILE_SIZE=ROWS_TILE_SIZE,
            D_TILE_SIZE=D_TILE_SIZE,
        )

        return y.view(output_dims)

    # 这里你应该为反向传播写一个 triton 核，而不是普通的 PyTorch！
    @staticmethod
    def backward(ctx, grad_output):
        # 取回保存的张量
        x, weight = ctx.saved_tensors

        # 把 grad_output 重塑成和 forward 匹配的形状
        grad_output_flat = grad_output.reshape(-1)

        # 关于 weight 的梯度：对所有样本求和
        # d/dw (w^T x) = x
        # 所以 grad_weight = sum_i grad_output[i] * x[i]
        grad_weight = (grad_output_flat[:, None] * x).sum(dim=0)

        # 关于 x 的梯度：广播 weight
        # d/dx (w^T x) = w
        # 所以 grad_x = grad_output * w
        grad_x = grad_output_flat[:, None] * weight[None, :]

        # 把 grad_x 重塑回原来的形状
        grad_x = grad_x.view(*ctx.output_dims, -1)

        return grad_x, grad_weight

In [ ]:
class LinearRegressionTriton(torch.nn.Module):
    """
    Linear regression using the custom Triton weighted sum kernel.

    Model: y = w^T x + b
    """

    def __init__(self, input_dim: int):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.randn(input_dim, device="cuda") * 0.01)
        self.bias = torch.nn.Parameter(torch.zeros(1, device="cuda"))

    def forward(self, x):
        # x: (batch_size, input_dim)
        # 使用我们自定义的加权和函数
        return WeightedSumFunc.apply(x, self.weight) + self.bias


### 端到端例子：用自定义 Triton 核做线性回归

**实验概述：** 这个实验演示如何把自定义 Triton GPU 核集成到真实的 PyTorch 训练流水线中。我们将训练一个线性回归模型（`y = w^T x + b`），其中前向传播用我们自定义的 `weighted_sum` Triton 核，而不是 PyTorch 的内置操作。

**我们要做什么：**
1. 用已知的真实权重生成合成回归数据
2. 创建一个 `LinearRegressionTriton` 模型，使用我们自定义的 `WeightedSumFunc` autograd 函数
3. 用标准的 PyTorch 优化（带 MSE 损失的 SGD）训练模型
4. 把学到的参数和生成数据的真实参数对比

**关键收获：** 这展示了如何编写生产就绪的 GPU 核，无缝集成到 PyTorch 的 autograd 系统中，让你可以用优化的自定义实现替换标准操作，同时仍然使用熟悉的训练循环。


In [ ]:


def generate_regression_data(n_samples=1000, input_dim=128, noise_std=0.1, seed=42):
    """
    Generate synthetic linear regression data.

    Returns:
        X: (n_samples, input_dim) feature matrix
        y: (n_samples,) continuous target values
        true_weight: (input_dim,) true weight vector used for generation
        true_bias: (1,) true bias value used for generation
    """
    torch.manual_seed(seed)

    # 生成随机特征
    X = torch.randn(n_samples, input_dim, device="cuda")

    # 创建用于生成数据的真实权重
    true_weight = torch.randn(input_dim, device="cuda")
    true_bias = torch.randn(1, device="cuda")

    # 生成目标值：y = w^T x + b + noise
    y = X @ true_weight + true_bias

    # 加高斯噪声
    noise = torch.randn(n_samples, device="cuda") * noise_std
    y = y + noise

    return X, y, true_weight, true_bias


def train_linear_regression(
    model, X_train, y_train, X_val, y_val, epochs=100, lr=0.01, batch_size=64
):
    """
    Train the linear regression model.

    Args:
        model: LinearRegressionTriton model
        X_train, y_train: Training data
        X_val, y_val: Validation data
        epochs: Number of training epochs
        lr: Learning rate
        batch_size: Batch size for training

    Returns:
        train_losses: List of training losses (MSE) per epoch
        val_losses: List of validation losses (MSE) per epoch
        val_r2_scores: List of validation R² scores per epoch
    """
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    train_losses = []
    val_losses = []
    val_r2_scores = []

    n_batches = (len(X_train) + batch_size - 1) // batch_size

    for epoch in range(epochs):
        # 训练
        model.train()
        epoch_loss = 0.0

        # 打乱数据
        perm = torch.randperm(len(X_train), device="cuda")
        X_train_shuffled = X_train[perm]
        y_train_shuffled = y_train[perm]

        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, len(X_train))

            X_batch = X_train_shuffled[start_idx:end_idx]
            y_batch = y_train_shuffled[start_idx:end_idx]

            # 前向传播
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # 反向传播
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / n_batches
        train_losses.append(avg_train_loss)

        # 验证
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val).item()
            val_losses.append(val_loss)

            # 计算 R² 分数
            ss_res = ((y_val - y_val_pred) ** 2).sum()
            ss_tot = ((y_val - y_val.mean()) ** 2).sum()
            r2_score = 1 - (ss_res / ss_tot)
            val_r2_scores.append(r2_score.item())

        # 每 10 个 epoch 打印进度
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1}/{epochs}: "
                f"Train Loss = {avg_train_loss:.4f}, "
                f"Val Loss = {val_loss:.4f}, "
                f"Val R² = {r2_score.item():.4f}"
            )

    return train_losses, val_losses, val_r2_scores



In [ ]:
def main():
    """
    Main function to demonstrate linear regression with custom Triton kernel.
    """
    print("=" * 80)
    print("Linear Regression with Custom Triton Weighted Sum Kernel")
    print("=" * 80)

    # 检查 CUDA 是否可用
    if not torch.cuda.is_available():
        print("ERROR: CUDA is not available. This example requires a GPU.")
        return

    # 超参数
    n_train = 800
    n_val = 200
    input_dim = 128
    epochs = 100
    lr = 0.01
    batch_size = 64
    noise_std = 0.1

    print(f"\nDataset configuration:")
    print(f"  Training samples: {n_train}")
    print(f"  Validation samples: {n_val}")
    print(f"  Input dimension: {input_dim}")
    print(f"  Noise std: {noise_std}")
    print(f"\nTraining configuration:")
    print(f"  Epochs: {epochs}")
    print(f"  Learning rate: {lr}")
    print(f"  Batch size: {batch_size}")
    print()

    # 生成数据
    print("Generating synthetic linear regression data...")
    X, y, true_weight, true_bias = generate_regression_data(
        n_samples=n_train + n_val, input_dim=input_dim, noise_std=noise_std
    )

    # 分成训练和验证
    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:], y[n_train:]

    print(f"Training set: X shape = {X_train.shape}, y shape = {y_train.shape}")
    print(f"Validation set: X shape = {X_val.shape}, y shape = {y_val.shape}")
    print(f"\nTrue parameters:")
    print(f"  True weight norm: {true_weight.norm().item():.6f}")
    print(f"  True bias: {true_bias.item():.6f}")
    print()

    # 初始化模型
    print("Initializing linear regression model with Triton kernel...")
    model = LinearRegressionTriton(input_dim=input_dim)
    model = torch.compile(model)
    print(
        f"Model parameters: weight shape = {model.weight.shape}, bias shape = {model.bias.shape}"
    )
    print()

    # 训练模型
    print("Starting training...\n")
    train_losses, val_losses, val_r2_scores = train_linear_regression(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
    )

    # 最终评估
    print("\n" + "=" * 80)
    print("Training Complete!")
    print("=" * 80)
    print(f"Final Training Loss (MSE): {train_losses[-1]:.4f}")
    print(f"Final Validation Loss (MSE): {val_losses[-1]:.4f}")
    print()

    # 把学到的参数和真实参数对比
    print("=" * 80)
    print("Parameter Comparison: Learned vs True")
    print("=" * 80)

    learned_weight = model.weight.data
    learned_bias = model.bias.data

    # 计算各种对比指标
    weight_diff = learned_weight - true_weight
    weight_mse = (weight_diff**2).mean().item()
    weight_mae = weight_diff.abs().mean().item()
    

    bias_diff = (learned_bias - true_bias).abs().item()


    print(f"\nWeight Statistics:")
    print(f"  True weight norm:        {true_weight.norm().item():.6f}")
    print(f"  Learned weight norm:     {learned_weight.norm().item():.6f}")
    print(f"  Weight MSE:              {weight_mse:.6f}")
    print(f"  Weight MAE:              {weight_mae:.6f}")
    
    print(f"\nBias Statistics:")
    print(f"  True bias:               {true_bias.item():.6f}")
    print(f"  Learned bias:            {learned_bias.item():.6f}")
    print(f"  Bias absolute difference: {bias_diff:.6f}")
    print()

    print("=" * 80)
    print("Example completed successfully!")
    print("=" * 80)

In [ ]:
main()